In [ ]:
# Import necessary libraries

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import cvxpy as cp

In [ ]:
def rotation_matrix(theta):
    return np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta), np.cos(theta)]
    ])

In [ ]:
T = 1000
n = 2

def new_dataset(T):
    v = np.array([0., 2.]).reshape(-1,1)
    ideal_V_set = []

    for t in range(T):
        theta = np.random.uniform(-np.pi/6, np.pi/6)
        R = rotation_matrix(theta)
        ideal_V_set.append(R @ v)
    
    V_upper_constraints = []
    for t in range(T):
        constraint_vec = np.random.uniform(0.1, 3., size=2).reshape(-1,1)
        V_upper_constraints.append(constraint_vec)
    
    V_lower_constraints = []
    for t in range(T):
        constraint_vec = np.random.uniform(-3, -0.1, size=2).reshape(-1,1)
        V_lower_constraints.append(constraint_vec)
    
    return ideal_V_set, V_upper_constraints, V_lower_constraints

In [ ]:
data = new_dataset(T)

# Algorithms

## CLASP-I

In [ ]:
class CLASP_I:
    
    def __init__(self, x_0, T, assumptions_param):
        
        self.x_0 = x_0        # Initial guess
        self.n = x_0.shape[0] # Optmization variable in R^n
        
        
    def projection_step(self, data):
        
        x, v_t_safe, A, idx = data
        
        y = cp.Variable((n,1))
        cost = (1/2) * cp.norm(y - x)**2

        constraints = [cp.norm(y) <= 4.]
        constraints.append(A[idx] @ y <= v_t_safe[idx])

        prob = cp.Problem(cp.Minimize(cost), constraints)
        prob.solve()

        return y.value.copy()
    
    def run(self, dataset):
        
        # Start with initial guess
        x_pred = self.x_0.copy()
        
        # Save predictions and gradients
        Preds = [x_pred.flatten()]
        f_t_grads = np.zeros(x_pred.shape)
        
        # Obtain data
        Vs, Vs_upper_safe, Vs_lower_safe = dataset
        
        for t in range(T):
            
            # Observe f_t and g_t
            v_t, v_upper, v_lower = Vs[t], Vs_upper_safe[t], Vs_lower_safe[t]
            f_t = 1/2 * np.linalg.norm(x_pred - v_t)**2
            
            v_t_safe = np.concatenate((v_upper, -v_lower))
            A = np.concatenate((np.identity(n), -np.identity(n)), axis=0)
            
            g_t = A @ x_pred - v_t_safe
            g_t_idx = np.argmax(g_t)
            g_t = max(g_t[g_t_idx,0], 0.0)
            
            # Construct gradient of cost function
            f_grad = x_pred - v_t
            
            # Gradient step
            eta_t = 1/np.sqrt(t+1)
            x_pred = x_pred - (1/np.sqrt(t+1)) * f_grad
            
            # Projection Step
            data = (x_pred, v_t_safe, A, g_t_idx)
            x_pred = self.projection_step(data)
            
            ## Save decision
            Preds.append(x_pred.flatten())
        
        # Return last decision and set of decisions
        return x_pred, Preds

## CLASP-F

In [ ]:
class CLASP_F:
    
    def __init__(self, x_0, T, assumptions_param):
        
        self.x_0 = x_0        # Initial guess
        self.n = x_0.shape[0] # Optmization variable in R^n
        
        
    def projection_step(self, x):
        
        if np.linalg.norm(x) > 4:
            x = x / np.linalg.norm(x) * 4

        return x.copy()
    
    def run(self, dataset):
        
        # Start with initial guess
        x_pred = self.x_0.copy()
        
        # Save predictions and gradients
        Preds = [x_pred.flatten()]
        f_t_grads = np.zeros(x_pred.shape)
        
        # Obtain data
        Vs, Vs_upper_safe, Vs_lower_safe = dataset
        
        for t in range(T):
            
            # Observe f_t and g_t
            v_t, v_upper, v_lower = Vs[t], Vs_upper_safe[t], Vs_lower_safe[t]
            f_t = 1/2 * np.linalg.norm(x_pred - v_t)**2
            
            v_t_safe = np.concatenate((v_upper, -v_lower))
            A = np.concatenate((np.identity(n), -np.identity(n)), axis=0)
            
            g_t = A @ x_pred - v_t_safe
            g_t_idx = np.argmax(g_t)
            g_t = max(g_t[g_t_idx,0], 0.0)
            
            # Construct gradient of cost and constraint functions
            g_grad = np.zeros((A.shape[1],1))
            if g_t > 0.0:
                g_grad = A[g_t_idx].reshape(-1,1)
            
            # Intermediate step
            bar_x = x_pred
            if g_t > 0.0:
                bar_x = bar_x - (g_t / np.linalg.norm(g_grad)**2) * g_grad
            bar_x = self.projection_step(bar_x)
            
            # Gradient step
            f_grad = x_pred - v_t
            eta_t = 1/np.sqrt(t+1)
            bar_x = bar_x - eta_t * f_grad
            
            # Projection Step
            x_pred = self.projection_step(bar_x)
            
            ## Save decision
            Preds.append(x_pred.flatten())
        
        # Return last decision and set of decisions
        return x_pred, Preds

## AdaGrad

In [ ]:
class adv_AdaGrad:
    
    def __init__(self, x_0, T, assumptions_param):
        
        self.x_0 = x_0        # Initial guess
        self.n = x_0.shape[0] # Optmization variable in R^n
        
        # Assumptions parameters:
        # F denotes the Lipschitz constant of the cost functions
        # G denotes the Lipschitz constant of the constraint functions
        # D denotes the diameter of the static domain set
        self.F, self.G, self.D = assumptions_param
        
        # Algorithm parameters
        self.V = 1
        self.G_temp = max(self.F, self.G)
        self.alpha = 1 / (2 * self.G_temp * self.D)
        self.lameda = 1 / (2 * np.sqrt(T))
        
        self.Q = 0.0
    
    
    def _Phi_grad(self, x):
        # Derivative of the non-decreasing convex Lyapunov function $\Phi(x) = x^2$
        return self.lameda * np.exp(self.lameda * x)
    
    
    def regret_bound(self, t):
        return 2 * self.G_temp * self.D * (np.sqrt(t) + 1)
    
    
    def ccv_bound(self, t):
        return 6.6 * self.G_temp * self.D * (2 + np.log(t)) * np.sqrt(t)
    
    
    def run(self, dataset):
        
        # Start with initial guess
        x_pred = self.x_0.copy()
        
        # Save predictions and gradients
        Preds = [x_pred.flatten()]
        f_t_grads = np.zeros(x_pred.shape)
        f_t_grads_norms = 0.0
        
        # Obtain data
        Vs, Vs_upper_safe, Vs_lower_safe = dataset
        
        for i in range(T):
            
            # Observe f_t and g_t
            v_t, v_upper, v_lower = Vs[i], Vs_upper_safe[i], Vs_lower_safe[i]
            f_t = 1/2 * np.linalg.norm(x_pred - v_t)**2
            
            v_t_safe = np.concatenate((v_upper, -v_lower))
            A = np.concatenate((np.identity(n), -np.identity(n)), axis=0)
            
            g_t = A @ x_pred - v_t_safe
            g_t_idx = np.argmax(g_t)
            g_t = max(g_t[g_t_idx,0], 0.0)
            
            # Update CCV
            self.Q = self.Q + self.alpha * g_t
            
            # Construct gradient of cost and constraint functions
            g_grad = np.zeros((A.shape[1],1))
            if g_t > 0.0:
                g_grad = A[g_t_idx].reshape(-1,1)
            
            f_grad = x_pred - v_t
        
            g_grad, f_grad = self.alpha * g_grad, self.alpha * f_grad

            f_hat_t_grad = self.V * f_grad + self._Phi_grad(self.Q) * g_grad
            f_t_grads_norms += np.linalg.norm(f_hat_t_grad)**2
            
            # AdaGrad step
            
            ## Update and project
            eta_t = (np.sqrt(2) * D) / (2 * np.sqrt(f_t_grads_norms))
        
            x_pred = x_pred - eta_t * f_hat_t_grad
            
            if np.linalg.norm(x_pred) > 4:
                x_pred = x_pred / np.linalg.norm(x_pred) * 4
            
            ## Save decision
            Preds.append(x_pred.flatten())
        
        # Return last decision and set of decisions
        return x_pred, Preds

## RECOO

In [ ]:
class adv_RECOO:
    
    def __init__(self, x_0, T, assumptions_param):
        
        self.x_0 = x_0        # Initial guess
        self.n = x_0.shape[0] # Optmization variable in R^n
        
        # Assumptions parameters:
        # F denotes the Lipschitz constant of the cost functions
        # G denotes the Lipschitz constant of the constraint functions
        # D denotes the diameter of the static domain set
        self.F, self.G, self.D = assumptions_param
        
        # Algorithm parameters
        self.G_temp = max(self.F, self.G)
        
        self.Q = 0.0
    
    
    def run(self, dataset):
        
        # Start with initial guess
        x_pred = self.x_0.copy()
        
        # Save predictions and gradients
        Preds = [x_pred.flatten()]
        
        # Obtain data
        Vs, Vs_upper_safe, Vs_lower_safe = dataset
        
        for t in range(T):
            
            # Parameters
            alpha_t = cp.sqrt(t+1)
            eta_t = np.sqrt(t+1)
            epsilon = 1e-3
            gamma_t = cp.power(t, 0.5 * epsilon)
            
            x_var = cp.Variable((self.n,1))
            
            # Observe f_t and g_t
            v_t, v_upper, v_lower = Vs[t], Vs_upper_safe[t], Vs_lower_safe[t]
            f_t = 1/2 * np.linalg.norm(x_pred - v_t)**2
            
            v_t_safe = np.concatenate((v_upper, -v_lower))
            A = np.concatenate((np.identity(n), -np.identity(n)), axis=0)
            
            g_t = A @ x_pred - v_t_safe
            g_t_idx = np.argmax(g_t)
            g_t = max(g_t[g_t_idx,0], 0.0)
            
            self.Q = max(self.Q + g_t, eta_t)
            
            
            f_grad = x_pred - v_t
            
            g_t_fun = A[g_t_idx] @ x_var - v_t_safe[g_t_idx]
            g_t_tilde = gamma_t * cp.maximum(g_t_fun[0], 0.0)
            
            constraints = [cp.norm(x_var) <= 4.]
            
            cost = f_grad.T @ x_var + self.Q * g_t_tilde + alpha_t * cp.norm(x_var - x_pred)**2
            
            prob = cp.Problem(cp.Minimize(cost), constraints)
            prob.solve()
            
            x_pred = x_var.value.copy()

            ## Save decision
            Preds.append(x_pred.flatten())
        
        # Return last decision and set of decisions
        return x_pred, Preds

## Switch

In [ ]:
class adv_Switch:
    
    def __init__(self, x_0, T, assumptions_param):
        
        self.x_0 = x_0        # Initial guess
        self.n = x_0.shape[0] # Optmization variable in R^n
        
        # Assumptions parameters:
        # F denotes the Lipschitz constant of the cost functions
        # G denotes the Lipschitz constant of the constraint functions
        # D denotes the diameter of the static domain set
        self.F, self.G, self.D = assumptions_param
        
        # Algorithm parameters
        self.G_temp = max(self.F, self.G)
        self.V = 1
        self.beta = 1 / (2 * self.G_temp * self.D)
        self.lameda = 1 / (2 * np.sqrt(T))
        
        self.Q = 0.0
    
    
    def _Phi_grad(self, x):
        # Derivative of the non-decreasing convex Lyapunov function $\Phi(x) = x^2$
        return self.lameda * np.exp(self.lameda * x)
    
    
    def _projection_step(self, data):
        
        x_var, A_set, b_set, x = data
        
        cost = (1/2) * cp.norm(x_var - x)**2

        constraints = [cp.norm(x_var) <= 4.]
        
        if len(A_set) > 0:
            constraints.append(A_set @ x_var <= b_set)

        prob = cp.Problem(cp.Minimize(cost), constraints)
        prob.solve(warm_start=True)

        return x_var.value.copy()
    
    
    def run(self, dataset):
        
        # Start with initial guess
        x_pred = self.x_0.copy()
        
        # Save predictions and gradients
        Preds = [x_pred.flatten()]
        f_t_grads = np.zeros(x_pred.shape)
        f_t_grads_norms = 0.0
        
        # Obtain data
        Vs, Vs_upper_safe, Vs_lower_safe = dataset
        
        # OGD Algorithm History Set
        x_var = cp.Variable(self.x_0.shape)
        A_set = np.array([]).reshape(0, self.x_0.shape[0])
        b_set = np.array([]).reshape(0,1)
        
        i = 0
        
        while i < T and self.Q <= np.sqrt(T) * np.log(T):
            # Observe f_t and g_t
            v_t, v_upper, v_lower = Vs[i], Vs_upper_safe[i], Vs_lower_safe[i]
            f_t = 1/2 * np.linalg.norm(x_pred - v_t)**2
            
            v_t_safe = np.concatenate((v_upper, -v_lower))
            A = np.concatenate((np.identity(n), -np.identity(n)), axis=0)
            
            g_t = A @ x_pred - v_t_safe
            g_t_idx = np.argmax(g_t)
            g_t = max(g_t[g_t_idx,0], 0.0)
            
            # Update CCV
            self.Q = self.Q + g_t
            
            ## Apply OGD-based algorithm
                
            # Gradient step
            f_grad = x_pred - v_t

            eta_t = self.D / (self.G_temp * np.sqrt(i+1))
            grad_step = x_pred - eta_t * f_grad

            # First projection
            data = (x_var, A_set, b_set, grad_step)
            projected_w = self._projection_step(data)

            # Second projection
            A_t, b_t = A[g_t_idx], v_t_safe[g_t_idx]
            A_set = np.vstack([A_set, A_t.reshape(1,-1)])
            b_set = np.vstack([b_set, b_t.reshape(1,-1)])

            data = (x_var, A_set, b_set, projected_w)
            x_pred = self._projection_step(data)
            
            Preds.append(x_pred.flatten())
            
            i+= 1
        
        # Switch to AdaGrad
        self.Q = 0
        while i < T:
            
            # Observe f_t and g_t
            v_t, v_upper, v_lower = Vs[i], Vs_upper_safe[i], Vs_lower_safe[i]
            f_t = 1/2 * np.linalg.norm(x_pred - v_t)**2
            
            v_t_safe = np.concatenate((v_upper, -v_lower))
            A = np.concatenate((np.identity(n), -np.identity(n)), axis=0)
            
            g_t = A @ x_pred - v_t_safe
            g_t_idx = np.argmax(g_t)
            g_t = max(g_t[g_t_idx,0], 0.0)
            
            # Update CCV
            self.Q = self.Q + self.beta * g_t
            
            ## Apply AdaGrad
            
            # Construct gradient of cost and constraint functions
            g_grad = np.zeros((A.shape[1],1))
            if g_t > 0.0:
                g_grad = A[g_t_idx].reshape(-1,1)
            
            f_grad = x_pred - v_t
        
            g_grad, f_grad = self.alpha * g_grad, self.alpha * f_grad

            f_hat_t_grad = self.V * f_grad + self._Phi_grad(self.Q) * g_grad
            f_t_grads_norms += np.linalg.norm(f_hat_t_grad)**2
            
            # AdaGrad step
            
            ## Update and project
            eta_t = (np.sqrt(2) * D) / (2 * np.sqrt(f_t_grads_norms))
        
            x_pred = x_pred - eta_t * f_hat_t_grad
            
            ## Save decision
            Preds.append(x_pred.flatten())
            
            i+=1
        
        # Return last decision and set of decisions
        return x_pred, Preds

## Frank-Wolfe

In [ ]:
class adv_FrankWolfe:
    
    def __init__(self, x_0, T, assumptions_param):
        
        self.x_0 = x_0        # Initial guess
        self.n = x_0.shape[0] # Optmization variable in R^n
        
        # Assumptions parameters:
        # F denotes the Lipschitz constant of the cost functions
        # G denotes the Lipschitz constant of the constraint functions
        # D denotes the diameter of the static domain set
        self.F, self.G, self.D = assumptions_param
        
        # Algorithm parameters
        self.G_temp = max(self.F, self.G)
        self.beta = 1 / ((2**6) * self.G_temp * self.D)
        self.lameda = 1 / (2 * (T**(3/4)))
        self.gamma = 1
        
        self.Q = 0.0
    
    
    def _Phi_grad(self, x):
        # Derivative of the non-decreasing convex Lyapunov function $\Phi(x) = x^2$
        return self.lameda * np.exp(self.lameda * x)
    
    
    def _optimization_step(self, data):
        
        f_t_grads, x_s_k, eta_t, s_k = data
        
        x_var = cp.Variable(f_t_grads[0].shape)
        
        F_grad = -2 * x_s_k
        for f_sup_grad in f_t_grads[s_k - 1:]:
            F_grad += eta_t * f_sup_grad
        
        cost = 2 * cp.square(cp.norm(x_var)) + F_grad.T @ x_var
        
        constraints = [cp.norm(x_var) <= 4]

        prob = cp.Problem(cp.Minimize(cost), constraints)
        prob.solve()

        return x_var.value.copy()
    
    
    def run(self, dataset):
        
        # Start with initial guess
        x_pred = self.x_0.copy()
        
        # Save predictions and gradients
        Preds = [x_pred.flatten()]
        f_t_grads = []
        f_t_grads_norms = 0.0
        
        # Obtain data
        Vs, Vs_upper_safe, Vs_lower_safe = dataset
        
        # Frank-Wolfe Auxiliary variables
        G_k = 1
        s_k = 1
        x_s_k = x_pred.copy()
        
        for i in range(T):
            # Observe f_t and g_t
            v_t, v_upper, v_lower = Vs[i], Vs_upper_safe[i], Vs_lower_safe[i]
            f_t = 1/2 * np.linalg.norm(x_pred - v_t)**2
            
            v_t_safe = np.concatenate((v_upper, -v_lower))
            A = np.concatenate((np.identity(n), -np.identity(n)), axis=0)
            
            g_t = A @ x_pred - v_t_safe
            g_t_idx = np.argmax(g_t)
            g_t = max(g_t[g_t_idx,0], 0.0)
            
            # Update CCV
            self.Q = self.Q + g_t
            
            ## Apply Frank-Wolfe-based algorithm
            
            while G_k < self.beta * self.G_temp * (self.gamma + self._Phi_grad(self.beta * self.Q)):
                G_k = G_k * 2
                s_k = i + 1
                x_s_k = x_pred.copy()
                
            # Construct gradient of cost and constraint functions
            g_grad = np.zeros((A.shape[1],1))
            if g_t > 0.0:
                g_grad = A[g_t_idx].reshape(-1,1)
            
            f_grad = x_pred - v_t
            
            eta_t = self.D / (2 * G_k * T**(3/4))
            
            f_tilde_grad = self.gamma * self.beta * f_grad + self._Phi_grad(self.beta * self.Q) * self.beta * g_grad
            f_t_grads.append(f_tilde_grad)
            
            data = (f_t_grads, x_s_k, eta_t, s_k)
            
            v_pred = self._optimization_step(data)
            sigma = 2 / np.sqrt(i + 1 - s_k + 1)
            
                
            x_pred = x_pred + sigma * (v_pred - x_pred)
            
            Preds.append(x_pred.flatten())
        
        # Return last decision and set of decisions
        return x_pred, Preds

# Run Single Experiment

In [ ]:
def compute_cost_and_ccv(dataset, Preds):
    Cost_sum = 0.0
    Cost_arr = []
    CCV_sum = 0.0
    CCV_arr = [0.0]
    CCV_2_sum = 0.0
    CCV_2_arr = [0.0]

    # Obtain data
    Vs, Vs_upper_safe, Vs_lower_safe = dataset

    for i in range(T):
        v_t, v_upper, v_lower = Vs[i], Vs_upper_safe[i], Vs_lower_safe[i]
        x_pred = Preds[i].reshape(-1,1)
        Cost_sum += 1/2 * np.linalg.norm(x_pred - v_t)**2
        Cost_arr.append(Cost_sum)
        
        v_t_safe = np.concatenate((v_upper, -v_lower))
        A = np.concatenate((np.identity(n), -np.identity(n)), axis=0)

        g_t = A @ x_pred - v_t_safe
        g_t_idx = np.argmax(g_t)
        g_t = max(g_t[g_t_idx,0], 0.0)
        
        CCV_sum += max(g_t, 0.0)
        CCV_arr.append(CCV_sum)
        
        CCV_2_sum += max(g_t, 0.0)**2
        CCV_2_arr.append(CCV_2_sum)
    
    return Cost_arr, CCV_arr, CCV_2_arr

In [ ]:
T = 1000

# Assumptions parameters
D = 8
F = 8
G = 1

# Generate dataset
dataset = new_dataset(T)

# CLASP-I

In [ ]:
assumptions_param = (F, G, D)

#Initial guess
x_0 = np.array([0., 2.]).reshape(-1,1)


CLASP_I_alg = CLASP_I(x_0, T, assumptions_param)
_, Preds = CLASP_I_alg.run(dataset)

In [ ]:
Preds_dict = {}
Preds_dict["CLASP_I"] = Preds

In [ ]:
# Compute regret and CCV
Cost_CLASP_I, CCV_CLASP_I, CCV_2_CLASP_I = compute_cost_and_ccv(dataset, Preds)

In [ ]:
def plot_regret(regrets, labels):
    
    # Plot list of regrets
    if len(regrets) != len(labels):
        raise Exception("Length of regrets must be the same as labels")
    
    ts = np.linspace(1,T,T)
    for i in range(len(regrets)):
        plt.plot(ts, regrets[i], label = labels[i], alpha=0.8)
    
    plt.ylabel('Cumulative loss')
    plt.xlabel('$t$ (iterations)')
    plt.yscale('symlog')
    plt.legend()
    

def plot_ccv(ccvs, labels, y_label = '$CCV_{T,1}(t)$'):
    # Plot list of CCV
    if len(ccvs) != len(labels):
        raise Exception("Length of ccvs must be the same as labels")
    
    ts = np.linspace(0,T,T+1)
    for i in range(len(ccvs)):
        plt.plot(ts, ccvs[i], label = labels[i], alpha=0.8)
    
    plt.ylabel(y_label)
    plt.xlabel('$t$ (iterations)')
    plt.yscale('symlog')
    plt.legend()

In [ ]:
# Plot obtained cost at each round
plot_regret([Cost_CLASP_I], ['CLASP-I'])

In [ ]:
# Plot obtained CCV
plot_ccv([CCV_CLASP_I], 
         ['CLASP-I'])

In [ ]:
# Plot obtained CCV
plot_ccv([CCV_2_CLASP_I], 
         ['CLASP-I'], y_label='$CCV_{T,2}(t)$')

# AdaGrad

In [ ]:
AdaGrad_alg = adv_AdaGrad(x_0, T, assumptions_param)
_, Preds = AdaGrad_alg.run(dataset)

In [ ]:
Preds_dict["AdaGrad"] = Preds

In [ ]:
# Compute regret and CCV
Cost_AdaGrad, CCV_AdaGrad, CCV_2_AdaGrad = compute_cost_and_ccv(dataset, Preds)

In [ ]:
# Plot obtained cost at each round
plot_regret([Cost_CLASP_I, Cost_AdaGrad], 
            ['CLASP-I', 'AdaGrad'])

In [ ]:
# Plot obtained CCV
plot_ccv([CCV_CLASP_I, CCV_AdaGrad], 
         ['CLASP-I', 'AdaGrad'])

In [ ]:
# Plot obtained CCV
plot_ccv([CCV_2_CLASP_I, CCV_2_AdaGrad], 
         ['CLASP-I', 'AdaGrad'], y_label='$CCV_{T,2}(t)$')

## Run RECOO

In [ ]:
RECOO_alg = adv_RECOO(x_0, T, assumptions_param)
_, Preds = RECOO_alg.run(dataset)

In [ ]:
Preds_dict["RECOO"] = Preds

In [ ]:
# Compute regret and CCV
Cost_RECOO, CCV_RECOO, CCV_2_RECOO = compute_cost_and_ccv(dataset, Preds)

In [ ]:
# Plot obtained cost at each round
plot_regret([Cost_CLASP_I, Cost_AdaGrad, Cost_RECOO], 
            ['CLASP-I', 'AdaGrad', 'RECOO'])

In [ ]:
# Plot obtained CCV
plot_ccv([CCV_CLASP_I, CCV_AdaGrad, CCV_RECOO], 
         ['CLASP-I', 'AdaGrad', 'RECOO'])

In [ ]:
# Plot obtained CCV
plot_ccv([CCV_2_CLASP_I, CCV_2_AdaGrad, CCV_2_RECOO], 
         ['CLASP-I', 'AdaGrad', 'RECOO'], y_label='$CCV_{T,2}(t)$')

## Run Switch

In [ ]:
Switch_alg = adv_Switch(x_0, T, assumptions_param)
_, Preds = Switch_alg.run(dataset)

In [ ]:
Preds_dict["Switch"] = Preds

In [ ]:
# Compute regret and CCV
Cost_Switch, CCV_Switch, CCV_2_Switch = compute_cost_and_ccv(dataset, Preds)

In [ ]:
# Plot obtained cost at each round
plot_regret([Cost_CLASP_I, Cost_AdaGrad, Cost_RECOO, Cost_Switch], 
            ['CLASP-I', 'AdaGrad', 'RECOO', 'Switch'])

In [ ]:
# Plot obtained CCV
plot_ccv([CCV_CLASP_I, CCV_AdaGrad, CCV_RECOO, CCV_Switch], 
         ['CLASP-I', 'AdaGrad', 'RECOO', 'Switch'])

In [ ]:
# Plot obtained CCV
plot_ccv([CCV_2_CLASP_I, CCV_2_AdaGrad, CCV_2_RECOO, CCV_2_Switch], 
         ['CLASP-I', 'AdaGrad', 'RECOO', 'Switch'], y_label='$CCV_{T,2}(t)$')

## Run Frank-Wolfe

In [ ]:
FW_alg = adv_FrankWolfe(x_0, T, assumptions_param)
_, Preds = FW_alg.run(dataset)

In [ ]:
Preds_dict["FW"] = Preds

In [ ]:
# Compute regret and CCV
Cost_FW, CCV_FW, CCV_2_FW = compute_cost_and_ccv(dataset, Preds)

In [ ]:
# Plot obtained cost at each round
plot_regret([Cost_CLASP_I, Cost_AdaGrad, Cost_RECOO, Cost_Switch, Cost_FW], 
            ['CLASP-I', 'AdaGrad', 'RECOO', 'Switch', 'Frank-Wolfe'])

In [ ]:
# Plot obtained CCV
plot_ccv([CCV_CLASP_I, CCV_AdaGrad, CCV_RECOO, CCV_Switch, CCV_FW], 
         ['CLASP-I', 'AdaGrad', 'RECOO', 'Switch', 'Frank-Wolfe'])

In [ ]:
# Plot obtained CCV
plot_ccv([CCV_2_CLASP_I, CCV_2_AdaGrad, CCV_2_RECOO, CCV_2_Switch, CCV_2_FW], 
         ['CLASP-I', 'AdaGrad', 'RECOO', 'Switch', 'Frank-Wolfe'], y_label='$CCV_{T,2}(t)$')

## CLASP-F

In [ ]:
CLASP_F_alg = CLASP_F(x_0, T, assumptions_param)
_, Preds = CLASP_F_alg.run(dataset)

In [ ]:
Preds_dict["CLASP_F"] = Preds

In [ ]:
# Compute regret and CCV
Cost_CLASP_F, CCV_CLASP_F, CCV_2_CLASP_F = compute_cost_and_ccv(dataset, Preds)

In [ ]:
# Plot obtained cost at each round
plot_regret([Cost_CLASP_I, Cost_AdaGrad, Cost_RECOO, Cost_Switch, Cost_FW, Cost_CLASP_F], 
            ['CLASP-I', 'AdaGrad', 'RECOO', 'Switch', 'Frank-Wolfe', 'CLASP-F'])

In [ ]:
# Plot obtained CCV
plot_ccv([CCV_CLASP_I, CCV_AdaGrad, CCV_RECOO, CCV_Switch, CCV_FW, CCV_CLASP_F], 
         ['CLASP-I', 'AdaGrad', 'RECOO', 'Switch', 'Frank-Wolfe', 'CLASP-F'])

In [ ]:
# Plot obtained CCV
plot_ccv([CCV_2_CLASP_I, CCV_2_AdaGrad, CCV_2_RECOO, CCV_2_Switch, CCV_2_FW, CCV_2_CLASP_F], 
         ['CLASP-I', 'AdaGrad', 'RECOO', 'Switch', 'Frank-Wolfe', 'CLASP-F'], y_label='$CCV_{T,2}(t)$')

# Run Multiple Experiments

In [ ]:
S = 50    # Run 50 trials

n = 2
T = 1000

# Assumptions parameters
D = 8
F = 8
G = 1
assumptions_param = (F, G, D)

Cost_dict = {'CLASP_F': [], 'FW': [], 'Switch': [], 'RECOO': [], 'AdaGrad': [], 'CLASP_I': []}
CCV_dict = {'CLASP_F': [], 'FW': [], 'Switch': [], 'RECOO': [], 'AdaGrad': [], 'CLASP_I': []}
CCV_2_dict = {'CLASP_F': [], 'FW': [], 'Switch': [], 'RECOO': [], 'AdaGrad': [], 'CLASP_I': []}


for _ in tqdm(range(S)):
    dataset = new_dataset(T)

    #Initial guess
    x_0 = np.array([0., 2.]).reshape(-1,1)

    # Run Online Gradient Descent
    CLASP_I_alg = CLASP_I(x_0, T, assumptions_param)
    _, Preds = CLASP_I_alg.run(dataset)
    
    Cost_CLASP_I, CCV_CLASP_I, CCV_2_CLASP_I = compute_cost_and_ccv(dataset, Preds)
    Cost_dict['CLASP_I'].append(Cost_CLASP_I)
    CCV_dict['CLASP_I'].append(CCV_CLASP_I)
    CCV_2_dict['CLASP_I'].append(CCV_2_CLASP_I)
    
    # Run AdaGrad
    AdaGrad_alg = adv_AdaGrad(x_0, T, assumptions_param)
    _, Preds = AdaGrad_alg.run(dataset)
    
    Cost_AdaGrad, CCV_AdaGrad, CCV_2_AdaGrad = compute_cost_and_ccv(dataset, Preds)
    Cost_dict['AdaGrad'].append(Cost_AdaGrad)
    CCV_dict['AdaGrad'].append(CCV_AdaGrad)
    CCV_2_dict['AdaGrad'].append(CCV_2_AdaGrad)
    
    # Run CLASP-F
    CLASP_F_alg = CLASP_F(x_0, T, assumptions_param)
    _, Preds = CLASP_F_alg.run(dataset)
    
    Cost_CLASP_F, CCV_CLASP_F, CCV_2_CLASP_F = compute_cost_and_ccv(dataset, Preds)
    Cost_dict['CLASP_F'].append(Cost_CLASP_F)
    CCV_dict['CLASP_F'].append(CCV_CLASP_F)
    CCV_2_dict['CLASP_F'].append(CCV_2_CLASP_F)
    
    # Run RECOO
    RECOO_alg = adv_RECOO(x_0, T, assumptions_param)
    _, Preds = RECOO_alg.run(dataset)
    
    Cost_RECOO, CCV_RECOO,CCV_2_RECOO  = compute_cost_and_ccv(dataset, Preds)
    Cost_dict['RECOO'].append(Cost_RECOO)
    CCV_dict['RECOO'].append(CCV_RECOO)
    CCV_2_dict['RECOO'].append(CCV_2_RECOO)
    
    # Run Switch
    Switch_alg = adv_Switch(x_0, T, assumptions_param)
    _, Preds = Switch_alg.run(dataset)
    
    Cost_Switch, CCV_Switch,CCV_2_Switch  = compute_cost_and_ccv(dataset, Preds)
    Cost_dict['Switch'].append(Cost_Switch)
    CCV_dict['Switch'].append(CCV_Switch)
    CCV_2_dict['Switch'].append(CCV_2_Switch)
    
    # Run Frank-Wolfe
    FW_alg = adv_FrankWolfe(x_0, T, assumptions_param)
    _, Preds = FW_alg.run(dataset)
    
    Cost_FW, CCV_FW,CCV_2_FW  = compute_cost_and_ccv(dataset, Preds)
    Cost_dict['FW'].append(Cost_FW)
    CCV_dict['FW'].append(CCV_FW)
    CCV_2_dict['FW'].append(CCV_2_FW)

In [ ]:
SIZE = 18
plt.rc('axes', titlesize=SIZE)     # fontsize of the axes title
plt.rc('axes', labelsize=SIZE)    # fontsize of the x and y labels
plt.rc('xtick', labelsize=SIZE)    # fontsize of the tick labels
plt.rc('ytick', labelsize=SIZE)    # fontsize of the tick labels
plt.rc('legend', fontsize=SIZE-1)    # legend fontsize

In [ ]:
Cost_dict['CLASP_I'] = np.array(Cost_dict['CLASP_I'])
CCV_dict['CLASP_I'] = np.array(CCV_dict['CLASP_I'])
CCV_2_dict['CLASP_I'] = np.array(CCV_2_dict['CLASP_I'])

Cost_dict['AdaGrad'] = np.array(Cost_dict['AdaGrad'])
CCV_dict['AdaGrad'] = np.array(CCV_dict['AdaGrad'])
CCV_2_dict['AdaGrad'] = np.array(CCV_2_dict['AdaGrad'])

Cost_dict['CLASP_F'] = np.array(Cost_dict['CLASP_F'])
CCV_dict['CLASP_F'] = np.array(CCV_dict['CLASP_F'])
CCV_2_dict['CLASP_F'] = np.array(CCV_2_dict['CLASP_F'])

Cost_dict['RECOO'] = np.array(Cost_dict['RECOO'])
CCV_dict['RECOO'] = np.array(CCV_dict['RECOO'])
CCV_2_dict['RECOO'] = np.array(CCV_2_dict['RECOO'])

Cost_dict['Switch'] = np.array(Cost_dict['Switch'])
CCV_dict['Switch'] = np.array(CCV_dict['Switch'])
CCV_2_dict['Switch'] = np.array(CCV_2_dict['Switch'])

Cost_dict['FW'] = np.array(Cost_dict['FW'])
CCV_dict['FW'] = np.array(CCV_dict['FW'])
CCV_2_dict['FW'] = np.array(CCV_2_dict['FW'])


In [ ]:
mean_cost_AdaGrad = np.mean(Cost_dict['AdaGrad'], axis=0)
std_cost_AdaGrad = 2 * np.std(Cost_dict['AdaGrad'], axis=0)

mean_cost_CLASP_I = np.mean(Cost_dict['CLASP_I'], axis=0)
std_cost_CLASP_I = 2 * np.std(Cost_dict['CLASP_I'], axis=0)

mean_cost_CLASP_F = np.mean(Cost_dict['CLASP_F'], axis=0)
std_cost_CLASP_F = 2 * np.std(Cost_dict['CLASP_F'], axis=0)

mean_cost_RECOO = np.mean(Cost_dict['RECOO'], axis=0)
std_cost_RECOO = 2 * np.std(Cost_dict['RECOO'], axis=0)

mean_cost_Switch = np.mean(Cost_dict['Switch'], axis=0)
std_cost_Switch = 2 * np.std(Cost_dict['Switch'], axis=0)

mean_cost_FW = np.mean(Cost_dict['FW'], axis=0)
std_cost_FW = 2 * np.std(Cost_dict['FW'], axis=0)

In [ ]:
ts = np.linspace(1,T,T)

plt.plot(ts, mean_cost_AdaGrad, label='AdaGrad')
plt.fill_between(ts, mean_cost_AdaGrad-std_cost_AdaGrad, mean_cost_AdaGrad+std_cost_AdaGrad, alpha=0.35)

plt.plot(ts, mean_cost_CLASP_I, label='CLASP-I')
plt.fill_between(ts, mean_cost_CLASP_I-std_cost_CLASP_I, mean_cost_CLASP_I+std_cost_CLASP_I, alpha=0.35)

plt.plot(ts, mean_cost_CLASP_F, label='CLASP-F')
plt.fill_between(ts, mean_cost_CLASP_F-std_cost_CLASP_F, mean_cost_CLASP_F+std_cost_CLASP_F, alpha=0.35)

plt.plot(ts, mean_cost_RECOO, label='RECOO')
plt.fill_between(ts, mean_cost_RECOO-std_cost_RECOO, mean_cost_RECOO+std_cost_RECOO, alpha=0.35)

plt.plot(ts, mean_cost_FW, label='Frank-Wolfe')
plt.fill_between(ts, mean_cost_FW-std_cost_FW, mean_cost_FW+std_cost_FW, alpha=0.35)

plt.plot(ts, mean_cost_Switch, label='Switch')
plt.fill_between(ts, mean_cost_Switch-std_cost_Switch, mean_cost_Switch+std_cost_Switch, alpha=0.35)

plt.ylabel('Cumulative loss')
plt.xlabel('$t$ (iterations)')
plt.yscale('log')
plt.legend(loc='lower right')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.savefig('Cost_regression_drone.pdf', format='pdf', bbox_inches='tight')

In [ ]:
mean_ccv_AdaGrad = np.mean(CCV_dict['AdaGrad'], axis=0)
std_ccv_AdaGrad = 2 * np.std(CCV_dict['AdaGrad'], axis=0)

mean_ccv_CLASP_I = np.mean(CCV_dict['CLASP_I'], axis=0)
std_ccv_CLASP_I = 2 * np.std(CCV_dict['CLASP_I'], axis=0)

mean_ccv_CLASP_F = np.mean(CCV_dict['CLASP_F'], axis=0)
std_ccv_CLASP_F = 2 * np.std(CCV_dict['CLASP_F'], axis=0)

mean_ccv_RECOO = np.mean(CCV_dict['RECOO'], axis=0)
std_ccv_RECOO = 2 * np.std(CCV_dict['RECOO'], axis=0)

mean_ccv_Switch = np.mean(CCV_dict['Switch'], axis=0)
std_ccv_Switch = 2 * np.std(CCV_dict['Switch'], axis=0)

mean_ccv_FW = np.mean(CCV_dict['FW'], axis=0)
std_ccv_FW = 2 * np.std(CCV_dict['FW'], axis=0)

In [ ]:
ts = np.linspace(0,T,T+1)
plt.plot(ts, mean_ccv_AdaGrad, label='AdaGrad')
plt.fill_between(ts, mean_ccv_AdaGrad-std_ccv_AdaGrad, mean_ccv_AdaGrad+std_ccv_AdaGrad, alpha=0.35)

plt.plot(ts, mean_ccv_CLASP_I, label='CLASP-I')
plt.fill_between(ts, mean_ccv_CLASP_I-std_ccv_CLASP_I, mean_ccv_CLASP_I+std_ccv_CLASP_I, alpha=0.35)

plt.plot(ts, mean_ccv_CLASP_F, label='CLASP-F')
plt.fill_between(ts, mean_ccv_CLASP_F-std_ccv_CLASP_F, mean_ccv_CLASP_F+std_ccv_CLASP_F, alpha=0.35)

plt.plot(ts, mean_ccv_RECOO, label='RECOO')
plt.fill_between(ts, mean_ccv_RECOO-std_ccv_RECOO, mean_ccv_RECOO+std_ccv_RECOO, alpha=0.35)

plt.plot(ts, mean_ccv_FW, label='Frank-Wolfe')
plt.fill_between(ts, mean_ccv_FW-std_ccv_FW, mean_ccv_FW+std_ccv_FW, alpha=0.35)

plt.plot(ts, mean_ccv_Switch, label='Switch')
plt.fill_between(ts, mean_ccv_Switch-std_ccv_Switch, mean_ccv_Switch+std_ccv_Switch, alpha=0.35)

plt.yscale('log')
plt.ylabel('$CCV_{t,1}$')
plt.xlabel('$t$ (iterations)')
#plt.legend()
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.savefig('CCV_{T,1}_regression_drone.pdf', format='pdf', bbox_inches='tight')

In [ ]:
mean_ccv_2_AdaGrad = np.mean(CCV_2_dict['AdaGrad'], axis=0)
std_ccv_2_AdaGrad = 2 * np.std(CCV_2_dict['AdaGrad'], axis=0)

mean_ccv_2_CLASP_I = np.mean(CCV_2_dict['CLASP_I'], axis=0)
std_ccv_2_CLASP_I = 2 * np.std(CCV_2_dict['CLASP_I'], axis=0)

mean_ccv_2_CLASP_F = np.mean(CCV_2_dict['CLASP_F'], axis=0)
std_ccv_2_CLASP_F = 2 * np.std(CCV_2_dict['CLASP_F'], axis=0)

mean_ccv_2_RECOO = np.mean(CCV_2_dict['RECOO'], axis=0)
std_ccv_2_RECOO = 2 * np.std(CCV_2_dict['RECOO'], axis=0)

mean_ccv_2_Switch = np.mean(CCV_2_dict['Switch'], axis=0)
std_ccv_2_Switch = 2 * np.std(CCV_2_dict['Switch'], axis=0)

mean_ccv_2_FW = np.mean(CCV_2_dict['FW'], axis=0)
std_ccv_2_FW = 2 * np.std(CCV_2_dict['FW'], axis=0)

In [ ]:
ts = np.linspace(0,T,T+1)
plt.plot(ts, mean_ccv_2_AdaGrad, label='AdaGrad')
plt.fill_between(ts, mean_ccv_2_AdaGrad-std_ccv_2_AdaGrad, mean_ccv_2_AdaGrad+std_ccv_2_AdaGrad, alpha=0.35)

plt.plot(ts, mean_ccv_2_CLASP_I, label='CLASP-I')
plt.fill_between(ts, mean_ccv_2_CLASP_I-std_ccv_2_CLASP_I, mean_ccv_2_CLASP_I+std_ccv_2_CLASP_I, alpha=0.35)

plt.plot(ts, mean_ccv_2_CLASP_F, label='CLASP-F')
plt.fill_between(ts, mean_ccv_2_CLASP_F-std_ccv_2_CLASP_F, mean_ccv_2_CLASP_F+std_ccv_2_CLASP_F, alpha=0.35)

plt.plot(ts, mean_ccv_2_RECOO, label='RECOO')
plt.fill_between(ts, mean_ccv_2_RECOO-std_ccv_2_RECOO, mean_ccv_2_RECOO+std_ccv_2_RECOO, alpha=0.35)

plt.plot(ts, mean_ccv_2_FW, label='Frank-Wolfe')
plt.fill_between(ts, mean_ccv_2_FW-std_ccv_2_FW, mean_ccv_2_FW+std_ccv_2_FW, alpha=0.35)

plt.plot(ts, mean_ccv_2_Switch, label='Switch')
plt.fill_between(ts, mean_ccv_2_Switch-std_ccv_2_Switch, mean_ccv_2_Switch+std_ccv_2_Switch, alpha=0.35)

plt.yscale('log')
plt.ylabel('$CCV_{t,2}$')
plt.xlabel('$t$ (iterations)')
#plt.legend(loc='lower right')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.savefig('CCV_{T,2}_regression_drone.pdf', format='pdf', bbox_inches='tight')